In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# os.listdir('/kaggle/input/competitions/llm-classification-finetuning')


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv


In [2]:
# !pip uninstall -y transformers
# !pip install transformers==5.17.0
# import transformers
# print(transformers.__version__)

In [3]:
df = pd.read_csv('/kaggle/input/competitions/llm-classification-finetuning/train.csv')
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [4]:
# print(df.iloc[0].to_dict())

for col in ['prompt', 'response_a', 'response_b']:
    lengths = df[col].astype(str).str.len()
    print(f"{col}: max={lengths.max()}, mean={lengths.mean():.0f}, p50={lengths.quantile(0.5):.0f}")

prompt: max=33056, mean=369, p50=96
response_a: max=54058, mean=1378, p50=1076
response_b: max=53830, mean=1386, p50=1086


In [5]:
from transformers import AutoTokenizer
import torch

checkpoint = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [6]:
# num_params = sum(p.numel() for p in model.parameters())
# print(f"{num_params/1e6:.1f}M params")

# sum(p.numel() for p in mdl.parameters() if p.requires_grad)

In [7]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

In [8]:
# dataset.to_pandas().head()

In [9]:
def to_text(s):
    parts = json.loads(s)
    return " ".join(x if x is not None else "" for x in parts)

# def preprocess(example):
#     p = tokenizer(to_text(example["prompt"]),     add_special_tokens=False)["input_ids"][:128]
#     a = tokenizer(to_text(example["response_a"]), add_special_tokens=False)["input_ids"][:180]
#     b = tokenizer(to_text(example["response_b"]), add_special_tokens=False)["input_ids"][:180]
#     sep = tokenizer.sep_token_id
#     cls = tokenizer.cls_token_id
#     ids = [cls] + p + [sep] + a + [sep] + b + [sep]
#     return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": example["label"]}

def preprocess(example):
    p = to_text(example["prompt"])
    a = to_text(example["response_a"])
    b = to_text(example["response_b"])

    enc_a = tokenizer(p, a, truncation=True, max_length=512)
    enc_b = tokenizer(p, b, truncation=True, max_length=512)

    return {
        "input_ids_a": enc_a["input_ids"],
        "attention_mask_a": enc_a["attention_mask"],
        "input_ids_b": enc_b["input_ids"],
        "attention_mask_b": enc_b["attention_mask"],
        "labels": example["label"],
    }

def add_label(example):
    if example["winner_model_a"] == 1: lbl = 0
    elif example["winner_model_b"] == 1: lbl = 1
    else: lbl = 2
    return {"label": lbl}

In [10]:
dataset = dataset.map(add_label)

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

In [11]:
# dataset = dataset.select(range(5000))
tokenized = dataset.map(preprocess, remove_columns=dataset.column_names)
split = tokenized.train_test_split(test_size=0.1, seed=42)
tokenized_train_ds = split["train"]
tokenized_val_ds   = split["test"]

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

In [12]:
# for i in range(10):
#     print(i, to_text(dataset[i]["prompt"])[:200])
#     print("---")

In [13]:
# print(tokenizer.decode(tokenized_train_ds[0]["input_ids_a"])[:2000])
# print(tokenizer.decode(tokenized_train_ds[0]["input_ids_b"])[:2000])

In [14]:
class SiameseCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    def __call__(self, features):
        a = self.tok.pad(
            [{"input_ids": f["input_ids_a"], "attention_mask": f["attention_mask_a"]} for f in features],
            return_tensors="pt",
        )
        b = self.tok.pad(
            [{"input_ids": f["input_ids_b"], "attention_mask": f["attention_mask_b"]} for f in features],
            return_tensors="pt",
        )
        labels = torch.tensor([f["labels"] for f in features])
        return {
            "input_ids_a": a["input_ids"],
            "attention_mask_a": a["attention_mask"],
            "input_ids_b": b["input_ids"],
            "attention_mask_b": b["attention_mask"],
            "labels": labels,
        }

In [15]:
# collator = SiameseCollator(tokenizer)
# batch = collator([tokenized_train_ds[i] for i in range(8)])
# for k, v in batch.items():
#     print(k, v.shape)

In [16]:
import torch.nn as nn
from transformers import AutoModel

class SiameseClassifier(nn.Module):
    def __init__(self, checkpoint, num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint, dtype=torch.float32)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(2 * hidden, num_labels)

    def mean_pool(self, last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()          # (B, L, 1)
        summed = (last_hidden * mask).sum(dim=1)             # (B, H)
        counts = mask.sum(dim=1).clamp(min=1e-6)             # (B, 1)
        return summed / counts

    def forward(self, input_ids_a, attention_mask_a, input_ids_b, attention_mask_b, labels=None):
        out_a = self.encoder(input_ids=input_ids_a, attention_mask=attention_mask_a).last_hidden_state
        out_b = self.encoder(input_ids=input_ids_b, attention_mask=attention_mask_b).last_hidden_state
        pool_a = self.mean_pool(out_a, attention_mask_a)
        pool_b = self.mean_pool(out_b, attention_mask_b)
        pooled = torch.cat([pool_a, pool_b], dim=-1)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}
        
    # def forward(self, input_ids_a, attention_mask_a, input_ids_b, attention_mask_b, labels=None):
    #     out_a = self.encoder(input_ids=input_ids_a, attention_mask=attention_mask_a).last_hidden_state[:, 0]
    #     out_b = self.encoder(input_ids=input_ids_b, attention_mask=attention_mask_b).last_hidden_state[:, 0]
    #     pooled = torch.cat([out_a, out_b], dim=-1)
    #     logits = self.classifier(pooled)

    #     loss = None
    #     if labels is not None:
    #         loss = nn.functional.cross_entropy(logits, labels)

    #     return {"loss": loss, "logits": logits}

In [17]:
from transformers import Trainer, TrainingArguments

data_collator = SiameseCollator(tokenizer)
model = SiameseClassifier(checkpoint, num_labels=3)

# batch = collator([tokenized_train_ds[i] for i in range(4)])
# out = model(**batch)
# print(out["logits"].shape)   # expect (4, 3)
# print(out["loss"])           # scalar

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
from scipy.special import softmax
from sklearn.metrics import log_loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)
    ll = log_loss(labels, probs, labels=[0, 1, 2])
    acc = (probs.argmax(axis=-1) == labels).mean()
    return {"log_loss": ll, "accuracy": acc}

In [19]:
training_args = TrainingArguments(
    output_dir="siamese_dir",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    fp16=True,
    logging_steps=50,
    report_to="none",
    metric_for_best_model="log_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

In [20]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=tokenized_train_ds,
                  eval_dataset=tokenized_val_ds,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Log Loss,Accuracy,Runtime,Samples Per Second,Steps Per Second
1,1.031081,1.017781,1.017781,0.483125,168.322000,34.149000,0.535000
2,0.965597,1.018052,1.018052,0.491301,168.050200,34.204000,0.536000


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=6468, training_loss=1.0082650296689555, metrics={'train_runtime': 8734.4648, 'train_samples_per_second': 11.845, 'train_steps_per_second': 0.741, 'total_flos': 0.0, 'train_loss': 1.0082650296689555, 'epoch': 2.0})

In [22]:
torch.save({
    "encoder_state": model.encoder.state_dict(),
    "classifier_state": model.classifier.state_dict(),
}, "/kaggle/working/siamese-small.pt")
tokenizer.save_pretrained("/kaggle/working/siamese-small-tok")

('/kaggle/working/siamese-small-tok/tokenizer_config.json',
 '/kaggle/working/siamese-small-tok/tokenizer.json')

In [23]:
# import torch
# print("model dtype:", next(model.parameters()).dtype)
# print("embedding dtype:", model.deberta.embeddings.word_embeddings.weight.dtype)
# print("classifier dtype:", model.classifier.weight.dtype)

In [24]:
# print(model.config.torch_dtype)
# print(model.config._name_or_path)

In [25]:
# import shutil, os
# shutil.rmtree("training_dir", ignore_errors=True)
# shutil.rmtree("/kaggle/working/deberta-xsmall-ft", ignore_errors=True)